[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Elheims/scikit-learn-cookbook/blob/main/02.%20Data%20Preprocessing/ch02_data_preprocessing.ipynb)

# Chapter 2: Pre-Model Workflow and Data Preprocessing

**Book:** *scikit-learn Cookbook, Third Edition* (John Sukup, Packt) — Chapter 2, pp. 36–67

**What you will learn in this chapter:**
- Why raw data quality (not the model) usually decides ML success: *garbage in, garbage out*
- How to handle missing data with `SimpleImputer`, `KNNImputer` and `IterativeImputer`
- How to scale features with `StandardScaler`, `MinMaxScaler` and `Normalizer` (and when *not* to)
- How to encode categorical variables with `OneHotEncoder`, `LabelEncoder` and `ColumnTransformer`
- How to chain preprocessing and modelling in a `Pipeline`, and why you split *before* transforming
- How to engineer features with `PolynomialFeatures`, `KBinsDiscretizer`, `RFE` and `SelectFromModel`
- How to put everything together in a comprehensive pipeline on the California Housing dataset

## Chapter summary

| Section | Core idea | Key API |
|---|---|---|
| Impact of raw data | Flawed data (missing, outliers, leakage…) degrades generalisation | — |
| Missing data | Fill gaps with a statistic, neighbours, or a regression model | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| Scaling | Put features on comparable scales | `StandardScaler`, `MinMaxScaler`, `Normalizer` |
| Encoding | Turn categories into numbers | `OneHotEncoder`, `LabelEncoder`, `ColumnTransformer` |
| Pipelines | One object = preprocessing + model, no leakage | `Pipeline`, `set_config(display="diagram")` |
| Feature engineering | Create informative features / drop useless ones | `PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel` |
| Practical exercise | Full pipeline on real data | California Housing |

**How this notebook is organised.** Each section has a summary, the book's code reproduced and executed, and a *Theory* cell.
Cells marked **Extension** are not in the book's code; I added them so a concept the book only describes can be seen working.
Places where the book's text is inaccurate are flagged with **Book correction** (collected again at the end).

In [89]:
import warnings
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

print("scikit-learn version:", sklearn.__version__)
print("NumPy version       :", np.__version__)
print("pandas version      :", pd.__version__)

scikit-learn version: 1.9.1
NumPy version       : 2.5.3
pandas version      : 3.0.6


## 1. The impact of raw data on model performance

**Summary.** ML algorithms learn patterns from data, so flawed input (missing values, outliers, irrelevant features, wrong
formats, leakage) lowers the model's ability to generalise to unseen data. The book lists the most common data issues and the
scikit-learn tool for each:

| Issue | Why it hurts | scikit-learn tool |
|---|---|---|
| Missing data | Most estimators refuse NaN or silently make wrong assumptions | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| Outliers | Distort means/variances and gradient-based fits | `RobustScaler`, Z-score filtering |
| Categorical variables | Models need numbers | `OneHotEncoder`, `LabelEncoder` |
| Different feature scales | Large-scale features dominate distance/gradient methods | `StandardScaler`, `MinMaxScaler` |
| Data leakage | Test information leaks into training → over-optimistic metrics | split first, then `Pipeline` |

Helpful building blocks: `Pipeline` (consistent transformations), `ColumnTransformer` (different steps per column type),
`sklearn.feature_selection` (`SelectFromModel`, `RFE`).

**Extension — see "garbage in, garbage out" for yourself.** Two quick experiments: (1) what a NaN does to an estimator;
(2) how much unscaled features hurt a distance-based model (KNN) on the *Wine* dataset.

In [90]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor

X_nan = np.array([[1.0], [2.0], [np.nan], [4.0]])
y_nan = np.array([1.0, 2.0, 3.0, 4.0])

try:
    LinearRegression().fit(X_nan, y_nan)
except ValueError as e:
    print("LinearRegression + NaN ->", type(e).__name__, ":", str(e).splitlines()[0])

# some estimators accept NaN natively (they declare it in their tags, see Chapter 1)
print("HistGradientBoostingRegressor + NaN -> fits fine:",
      HistGradientBoostingRegressor().fit(X_nan, y_nan) is not None)

LinearRegression + NaN -> ValueError : Input X contains NaN.
HistGradientBoostingRegressor + NaN -> fits fine: True


In [91]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_wine, y_wine = load_wine(return_X_y=True)
print("Feature scale ranges (min -> max of each feature's max):", X_wine.max(axis=0).min(), "->", X_wine.max(axis=0).max())

raw   = cross_val_score(KNeighborsClassifier(), X_wine, y_wine, cv=5).mean()
scaled = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier()), X_wine, y_wine, cv=5).mean()
print(f"KNN accuracy, raw features   : {raw:.3f}")
print(f"KNN accuracy, standardised   : {scaled:.3f}")

Feature scale ranges (min -> max of each feature's max): 0.66 -> 1680.0
KNN accuracy, raw features   : 0.691
KNN accuracy, standardised   : 0.949


### Theory
* **Garbage in, garbage out.** A model can only be as good as the information in its data. Preprocessing is not a one-time
  step: data changes over time (*data drift*), so the same cleaning has to be applied again in production.
* **Why generalisation suffers.** Noisy/biased training data makes the model fit patterns that do not hold for new data
  (overfitting) or learn a distorted picture of the true relationships (bias).
* **Outliers can also be signal.** For fraud or fault detection the rare events *are* what you want (see Chapter 11).
* **Data leakage** = information from outside the training set (typically the test set) influencing training. It gives
  metrics that look great in the notebook and fail in production. The cure is structural: split first, fit transformers
  only on training data, and wrap everything in a `Pipeline`.
* **Distance-based models** (KNN, SVM, K-Means) compare raw numbers, so a feature measured in the hundreds (proline in
  Wine) swamps a feature measured in fractions — that is what the accuracy gap above shows.

## 2. Handling missing data

**Summary.** Missing data arises from many sources — **human error, technical failures, data corruption** (and errors in data collection or system failures, as listed in Section 1). Most scikit-learn
estimators will not even run when NaNs are present. With a large enough dataset you can sometimes simply **drop the records** that contain missing values with little impact on the model, but this is not always viable; otherwise
*impute* (fill in) the gaps. Three imputers are covered:
`SimpleImputer` (a column statistic), `KNNImputer` (from similar rows) and `IterativeImputer` (regress each feature on the others).

### Getting ready — the book's toy dataset
20 samples × 10 uniform random features with about 20 % of the values removed at random.

In [92]:
import numpy as np
import pandas as pd

# 2. Create a larger sample dataset with missing values
np.random.seed(2024)  # For reproducibility
n_samples = 20
n_features = 10

# 3. Generate random data
data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
    for i in range(n_features)
}

# 4. Convert it to a DataFrame
df = pd.DataFrame(data)

# 5. Randomly introduce missing values (approximately 20% of the data)
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

# 6. Display the DataFrame with missing values
display(df)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,NaN,42.009814,34.680397,49.962259,NaN,NaN,89.954588,41.152421,NaN
1,69.910875,9.554215,6.436369,31.287816,37.966499,NaN,82.005649,NaN,75.797616,91.382492
2,NaN,96.090974,59.643269,84.710402,NaN,84.109027,NaN,70.288128,1.778343,4.117690
3,NaN,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,NaN,NaN,80.077973
4,20.501895,NaN,89.248639,67.655865,58.635861,78.225721,60.911562,NaN,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,NaN,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,NaN,22.966899,21.049022,57.358544,14.302591
7,NaN,NaN,89.538184,69.451294,NaN,47.885551,NaN,33.620401,99.685711,6.683138
8,47.384570,NaN,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,NaN,31.142866,28.865545,NaN,41.004459,41.460336,50.236236,NaN


In [93]:
print("Missing values per column:\n", df.isna().sum().to_string())
print(f"\nOverall missing: {df.isna().mean().mean():.1%}")

Missing values per column:
 Feature1     3
Feature2     5
Feature3     4
Feature4     4
Feature5     5
Feature6     5
Feature7     7
Feature8     6
Feature9     2
Feature10    3

Overall missing: 22.0%


### How to do it — `SimpleImputer()`
Replace each NaN with a statistic of its column: `"mean"`, `"median"`, `"most_frequent"` or `"constant"`.

In [94]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,53.864661,42.009814,34.680397,49.962259,57.822964,51.145011,89.954588,41.152421,48.400044
1,69.910875,9.554215,6.436369,31.287816,37.966499,57.822964,82.005649,50.715003,75.797616,91.382492
2,52.558605,96.090974,59.643269,84.710402,46.055883,84.109027,51.145011,70.288128,1.778343,4.117690
3,52.558605,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.715003,60.616723,80.077973
4,20.501895,53.864661,89.248639,67.655865,58.635861,78.225721,60.911562,50.715003,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.055883,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,57.822964,22.966899,21.049022,57.358544,14.302591
7,52.558605,53.864661,89.538184,69.451294,46.055883,47.885551,51.145011,33.620401,99.685711,6.683138
8,47.384570,53.864661,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.897507,31.142866,28.865545,57.822964,41.004459,41.460336,50.236236,48.400044


The book says Feature1's NaNs become the column mean **52.558605**. Check:

In [95]:
print("mean of observed Feature1 values:", df["Feature1"].mean())
print("learned statistics_ (first 3)   :", imputer.statistics_[:3])
print("rows 1-2 of Feature1 after imputation:", imputed_df.loc[[1, 2], "Feature1"].tolist())

mean of observed Feature1 values: 52.5586046974238
learned statistics_ (first 3)   : [52.5586047  53.86466147 52.89750669]
rows 1-2 of Feature1 after imputation: [69.91087476815825, 52.5586046974238]


### `KNNImputer()`
For each missing value, find the `n_neighbors` most similar rows (measured on the features that are present) and fill with
the **average** of those neighbours' values for that feature.

In [96]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)
knn_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,48.954043,42.009814,34.680397,49.962259,93.910386,52.549271,89.954588,41.152421,59.833678
1,69.910875,9.554215,6.436369,31.287816,37.966499,86.793468,82.005649,45.416191,75.797616,91.382492
2,67.752344,96.090974,59.643269,84.710402,73.338309,84.109027,59.012876,70.288128,1.778343,4.117690
3,47.880864,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,64.510281,39.726833,80.077973
4,20.501895,31.670912,89.248639,67.655865,58.635861,78.225721,60.911562,25.903612,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,59.716773,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,40.482911,22.966899,21.049022,57.358544,14.302591
7,47.629715,77.843853,89.538184,69.451294,32.753328,47.885551,55.683434,33.620401,99.685711,6.683138
8,47.384570,35.180639,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,29.188271,31.142866,28.865545,59.674651,41.004459,41.460336,50.236236,35.000820


> **Book correction.** The book says KNNImputer assigns "the majority label appearing among those values". That describes a
> KNN *classifier*. `KNNImputer` works on numeric features and fills each gap with the (by default uniform-weighted) **mean** of the
> neighbours' values, using the `nan_euclidean` distance.

### `IterativeImputer()`
Models each feature that has missing values as a regression target and the remaining features as inputs, cycling through the
features in a round-robin fashion until the estimates stabilise. It is still flagged *experimental*, so it must be enabled explicitly.

In [97]:
from sklearn.experimental import (
    enable_iterative_imputer
)  # Experimental feature requires loading
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer()
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(
    iterative_imputed_data, columns=df.columns)
iterative_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,54.365008,42.009814,34.680397,49.962259,58.680582,51.178102,89.954588,41.152421,48.292275
1,69.910875,9.554215,6.436369,31.287816,37.966499,60.070634,82.005649,50.710323,75.797616,91.382492
2,52.539676,96.090974,59.643269,84.710402,46.121290,84.109027,51.155371,70.288128,1.778343,4.117690
3,52.630591,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.680265,50.650877,80.077973
4,20.501895,53.198954,89.248639,67.655865,58.635861,78.225721,60.911562,50.691674,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.156310,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,55.923578,22.966899,21.049022,57.358544,14.302591
7,52.558131,53.942885,89.538184,69.451294,46.070477,47.885551,50.969808,33.620401,99.685711,6.683138
8,47.384570,54.204978,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.882614,31.142866,28.865545,55.978300,41.004459,41.460336,50.236236,48.429901


Compare the three imputations for the same cells (Feature1, rows that were NaN):

In [98]:
nan_rows = df.index[df["Feature1"].isna()]
pd.DataFrame({
    "Simple (mean)": imputed_df.loc[nan_rows, "Feature1"],
    "KNN (k=2)": knn_imputed_df.loc[nan_rows, "Feature1"],
    "Iterative": iterative_imputed_df.loc[nan_rows, "Feature1"],
})

,Simple (mean),KNN (k=2),Iterative
2,52.558605,67.752344,52.539676
3,52.558605,47.880864,52.630591
7,52.558605,47.629715,52.558131


Because this toy data is *independent uniform noise*, there is no relationship to exploit, so the three methods have little to
learn from. **Extension — a fair comparison on correlated data.** I take the real *Diabetes* feature matrix, hide 10 % of the
values at random, impute them, and measure the error against the true hidden values.

In [99]:
from sklearn.datasets import load_diabetes

X_full = load_diabetes(return_X_y=True)[0]
rng = np.random.default_rng(0)
hide = rng.random(X_full.shape) < 0.10
X_missing = X_full.copy()
X_missing[hide] = np.nan

results = {}
for name, imp in {
    "SimpleImputer(mean)":   SimpleImputer(strategy="mean"),
    "SimpleImputer(median)": SimpleImputer(strategy="median"),
    "KNNImputer(k=5)":       KNNImputer(n_neighbors=5),
    "IterativeImputer":      IterativeImputer(random_state=0),
}.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # IterativeImputer may warn that it stopped at max_iter
        X_hat = imp.fit_transform(X_missing)
    results[name] = np.sqrt(np.mean((X_hat[hide] - X_full[hide]) ** 2))

pd.Series(results, name="RMSE on hidden values").sort_values().round(4).to_frame()

,RMSE on hidden values
IterativeImputer,0.0313
KNNImputer(k=5),0.0380
SimpleImputer(mean),0.0465
SimpleImputer(median),0.0496


Lower is better. On correlated features the model-based imputers use the relationships between columns and beat the plain
column statistic, which is the book's argument for the more complex methods.

### There's more… choosing an imputation strategy
Factors the book lists: **data type** (numeric → mean/median, categorical → most frequent), **distribution** (skewed → median),
**missing-data pattern**, and **amount missing**.

> **Book correction.** The book writes "When **>5%** of your dataset contains missing values, `SimpleImputer()` is a good
> candidate", then "5–10 % → KNNImputer" and ">10 %" for the sparser case. The first threshold is clearly a typo for **<5 %**
> (a small amount of missingness), otherwise the three ranges overlap.

Also, tree-based models (decision trees, Random Forest) don't *require* imputation in the book's words; more precisely,
scikit-learn's `HistGradientBoosting*` and, in recent versions, `DecisionTree*`/`RandomForest*` accept NaN natively — as shown in the first section.

### Theory
* **Missingness mechanisms (Rubin).**
  *MCAR* — missing completely at random (independent of any data); *MAR* — depends on other observed variables;
  *MNAR* — depends on the missing value itself (e.g. high earners hide their income). Simple imputation is only
  unbiased for MCAR; MNAR needs domain-specific modelling.
* **SimpleImputer** keeps the column mean (or median) but **shrinks the variance** and weakens correlations, because every filled value is identical.
  Median is robust to outliers; most-frequent works for categories.
* **KNNImputer** distance uses only coordinates present in both rows (`nan_euclidean`):
  $d(x,y)=\sqrt{\dfrac{p}{p_{obs}}\sum_{j\in obs}(x_j-y_j)^2}$ with $p$ total and $p_{obs}$ jointly observed features.
  Features should be scaled first, otherwise large-scale features decide who the neighbours are.
* **IterativeImputer** is the idea behind MICE (multivariate imputation by chained equations): initialise NaNs with the mean, then for each feature
  fit a regressor (default `BayesianRidge`) on the other features and replace the NaNs by predictions; repeat up to `max_iter`
  (default 10) rounds. It is the most expensive and the most accurate on correlated data.
* **Fit on training data only**, like every transformer — otherwise the imputer leaks test information.
* `add_indicator=True` appends binary "was missing" columns, useful when missingness itself is informative.

### Hands-on workflow (from the book)
1. Load data and identify missing values. 2. Choose an imputation method. 3. Apply the imputer. 4. Evaluate with cross-validation
(the **Extension** above uses a hidden-value RMSE; Section 5 shows imputation inside a cross-validated pipeline).

## 3. Scaling techniques

**Summary.** Features measured on very different scales (age 0–100 vs income 0–100 000) distort algorithms that rely on distances
or gradients. The book distinguishes two terms that are often (wrongly) used interchangeably:

* **Standardization** (z-score) → mean 0, standard deviation 1.
* **Normalization** → change the range so that values lie in [0, 1] (the book's definition; in scikit-learn `Normalizer` is something different — see below).

Three scalers are shown, all applied to `iterative_imputed_df` from the previous section.

### `StandardScaler()`
$$z=\frac{x-\mu}{\sigma}$$

In [100]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(
    scaled_data, columns=iterative_imputed_df.columns)
scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.271931,0.019495,-0.373057,-0.894258,0.188761,0.031844,0.001905,1.683800,-0.780898,-0.003574
1,0.756048,-1.874939,-1.592196,-1.043864,-0.391744,0.085949,1.426871,-0.000534,0.652484,1.429056
2,-0.000939,1.783513,0.231259,1.311954,0.002887,1.021588,0.000854,0.839730,-2.409929,-1.472256
3,0.003022,-1.214477,1.056818,1.458037,-1.411839,1.531340,1.551267,-0.001824,-0.387917,1.053212
4,-1.397054,-0.029802,1.245866,0.559887,0.608499,0.792594,0.451822,-0.001334,0.210478,1.686279
5,-1.828276,0.969036,-1.125549,-2.186891,0.004582,-1.485267,-0.772566,1.831653,0.503915,-1.188904
6,0.878636,1.094467,-0.091017,0.042422,-1.929442,-0.075466,-1.302124,-1.273574,-0.110399,-1.133636
7,-0.000135,0.001649,1.255789,0.639061,0.000428,-0.388327,-0.007723,-0.734020,1.640810,-1.386962
8,-0.225584,0.012729,-0.935710,1.210940,1.323678,0.147955,-1.015275,0.653400,0.040472,-0.476999
9,-0.336923,-0.665377,-0.000435,-1.050256,-0.832163,-0.073336,-0.468360,-0.397536,-0.405072,0.001002


In [101]:
print(scaled_df.agg(["mean", "std"]).round(3).to_string())

      Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
mean    -0.000     0.000     0.000    -0.000     0.000    -0.000    -0.000    -0.000     0.000     -0.000
std      1.026     1.026     1.026     1.026     1.026     1.026     1.026     1.026     1.026      1.026


Means are 0 (up to floating-point error). The `std` rows read 1.026 rather than 1.000 because pandas uses the *sample* standard
deviation (`ddof=1`) while `StandardScaler` divides by the *population* one (`ddof=0`); both are legitimate.

### `MinMaxScaler()`
$$x'=\frac{x-x_{min}}{x_{max}-x_{min}}$$

In [102]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
minmax_scaled_data = minmax_scaler.fit_transform(
    iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(
    minmax_scaled_data,
    columns=iterative_imputed_df.columns
)
minmax_scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.603506,0.517824,0.380540,0.354639,0.569020,0.555896,0.435006,0.962767,0.402156,0.464988
1,0.721357,0.000000,0.000369,0.313594,0.413077,0.571920,0.833074,0.538604,0.756013,0.918562
2,0.537080,1.000000,0.568987,0.959922,0.519088,0.849027,0.434713,0.750206,0.000000,0.000000
3,0.538045,0.180530,0.826426,1.000000,0.139045,1.000000,0.867825,0.538279,0.499171,0.799569
4,0.197218,0.504349,0.885377,0.753589,0.681776,0.781206,0.560692,0.538402,0.646896,1.000000
5,0.092244,0.777371,0.145886,0.000000,0.519543,0.106575,0.218656,1.000000,0.719336,0.089710
6,0.751199,0.811657,0.468490,0.611621,0.000000,0.524114,0.070723,0.218016,0.567681,0.107208
7,0.537276,0.512946,0.888472,0.775311,0.518428,0.431454,0.432317,0.353891,1.000000,0.027004
8,0.482394,0.515975,0.205085,0.932208,0.873897,0.590284,0.150855,0.703283,0.604927,0.315101
9,0.455290,0.330621,0.496737,0.311840,0.294766,0.524745,0.303637,0.438627,0.494936,0.466437


In [103]:
print(minmax_scaled_df.agg(["min", "max"]).to_string())

     Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
min       0.0       0.0       0.0       0.0       0.0       0.0       0.0       0.0       0.0        0.0
max       1.0       1.0       1.0       1.0       1.0       1.0       1.0       1.0       1.0        1.0


### `Normalizer()`
Unlike the two scalers above (which work per **column**), `Normalizer` works per **row**: it rescales every sample so its
vector has unit norm. L2 (default) and L1 norms:
$$\lVert x\rVert_1=\sum_i |x_i| \qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}$$

In [104]:
from sklearn.preprocessing import Normalizer

normalizer = Normalizer()
normalized_data = normalizer.fit_transform(
    iterative_imputed_df)
normalized_df = pd.DataFrame(
    normalized_data,
    columns=iterative_imputed_df.columns
)
normalized_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339168,0.313579,0.242313,0.200037,0.288183,0.338471,0.295196,0.518860,0.237368,0.278551
1,0.376706,0.051482,0.034682,0.168591,0.204578,0.323683,0.441878,0.273247,0.408426,0.492404
2,0.264336,0.483450,0.300075,0.426192,0.232044,0.423166,0.257371,0.353631,0.008947,0.020717
3,0.243762,0.116607,0.387811,0.407684,0.078213,0.450213,0.392278,0.234729,0.234592,0.370886
4,0.095909,0.248868,0.417511,0.316499,0.274302,0.365945,0.284948,0.237139,0.304609,0.463686
5,0.068115,0.493382,0.128781,0.034471,0.296421,0.126535,0.221071,0.599823,0.463720,0.081177
6,0.460521,0.505281,0.318139,0.354119,0.039204,0.354133,0.145437,0.133292,0.363220,0.090570
7,0.274582,0.281816,0.467778,0.362837,0.240688,0.250170,0.266284,0.175644,0.520792,0.034915
8,0.265283,0.303467,0.143277,0.461427,0.411012,0.345225,0.163323,0.369203,0.341538,0.190645
9,0.321287,0.273524,0.379002,0.223196,0.206875,0.401188,0.293873,0.297140,0.360036,0.347090


In [105]:
print("L2 norm of every row (should all be 1):")
print(np.linalg.norm(normalized_df.values, axis=1).round(6))
print("\nL1 variant, row sums of |x|:")
print(np.abs(Normalizer(norm="l1").fit_transform(iterative_imputed_df)).sum(axis=1).round(6))

L2 norm of every row (should all be 1):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

L1 variant, row sums of |x|:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


> **Terminology note.** What the book calls *normalization* (squash into [0, 1]) is exactly `MinMaxScaler`.
> scikit-learn's `Normalizer` is *sample-wise* unit-norm scaling — a different idea. `Normalizer` is stateless: `fit` learns nothing.

### Extension — other scalers from the book's decision guide (Figure 2.17)
The chapter ends with a decision guide that also names `RobustScaler`, `MaxAbsScaler`, `PowerTransformer` and `QuantileTransformer`.
Here is what an outlier does to each:

In [106]:
from sklearn.preprocessing import RobustScaler, MaxAbsScaler, PowerTransformer, QuantileTransformer

x = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [100.0]])   # one big outlier
table = pd.DataFrame({
    "raw": x.ravel(),
    "Standard": StandardScaler().fit_transform(x).ravel(),
    "MinMax": MinMaxScaler().fit_transform(x).ravel(),
    "Robust": RobustScaler().fit_transform(x).ravel(),
    "MaxAbs": MaxAbsScaler().fit_transform(x).ravel(),
    "Power(Yeo-Johnson)": PowerTransformer().fit_transform(x).ravel(),
    "Quantile(normal)": QuantileTransformer(n_quantiles=6, output_distribution="normal").fit_transform(x).ravel(),
}).round(3)
table

,raw,Standard,MinMax,Robust,MaxAbs,Power(Yeo-Johnson),Quantile(normal)
0,1.0,-0.502,0.00,-1.0,0.01,-1.500,-5.199
1,2.0,-0.475,0.01,-0.6,0.02,-0.639,-0.842
2,3.0,-0.447,0.02,-0.2,0.03,-0.160,-0.253
3,4.0,-0.419,0.03,0.2,0.04,0.151,0.253
4,5.0,-0.392,0.04,0.6,0.05,0.372,0.842
5,100.0,2.235,1.00,38.6,1.00,1.776,5.199


`StandardScaler` and `MinMaxScaler` squeeze the five normal points together (the outlier dominates the mean/range);
`RobustScaler` (uses median and IQR) keeps them well spread; `PowerTransformer` and `QuantileTransformer` reshape the distribution.

**Extension — which models care about scaling?** KNN and SVM do; trees do not:

In [107]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

def cv(model): return cross_val_score(model, X_wine, y_wine, cv=5).mean()

rows = []
for name, est in {"KNN": KNeighborsClassifier(), "SVC": SVC(), "RandomForest": RandomForestClassifier(random_state=0)}.items():
    rows.append({"model": name,
                 "raw": cv(est),
                 "StandardScaler": cv(make_pipeline(StandardScaler(), est)),
                 "MinMaxScaler": cv(make_pipeline(MinMaxScaler(), est))})
pd.DataFrame(rows).set_index("model").round(3)

,raw,StandardScaler,MinMaxScaler
model,,,
KNN,0.691,0.949,0.950
SVC,0.663,0.983,0.978
RandomForest,0.983,0.983,0.983


KNN and SVC jump when scaled; Random Forest is unchanged because a split such as `x < 3.2` is invariant to monotonic rescaling
(the book: "tree-based methods ... work using the raw data values").

### Theory
* **Why scale?** (a) Distance-based methods (KNN, K-Means, SVM-RBF) — otherwise the large-range feature decides everything.
  (b) Gradient-based methods (linear/logistic regression, neural nets) — scaled features give a well-conditioned loss surface so gradient descent converges faster.
  (c) Regularised models (Ridge/Lasso, Chapter 5) — the penalty $\lambda\sum w_j^2$ treats all coefficients equally, which is only fair if features are on the same scale.
  (d) PCA (Chapter 3) — variance is scale-dependent.
* **StandardScaler** — best for roughly Gaussian data; sensitive to outliers (mean/std). Output is unbounded.
* **MinMaxScaler** — bounded output (or any range via `feature_range`); very sensitive to outliers because min/max define the range.
* **Normalizer** — row-wise; useful for text/TF-IDF vectors and cosine-similarity-style comparisons where direction matters, not magnitude.
* **RobustScaler** $\;x'=\dfrac{x-\mathrm{median}}{Q_3-Q_1}$ — outlier resistant. **MaxAbsScaler** divides by $\max|x|$, keeps zeros (good for sparse data).
* **Fit on training data only**; apply `transform` to the test data.

### Hands-on workflow (from the book)
Identify features that need scaling → pick scalers by data characteristics → apply each → train several models → compare with accuracy / precision /
recall / MSE. The model-by-scaler table above is exactly this workflow.

### The book's decision guide (Figure 2.17) as a table
| Question | Answer → choice |
|---|---|
| Has outliers? | Yes → `RobustScaler`, `QuantileTransformer` or `PowerTransformer`; No → check distribution |
| Distribution | Normal → `StandardScaler` / `MinMaxScaler`; Skewed → `PowerTransformer` (Yeo-Johnson / Box-Cox); Unknown or mixed → `QuantileTransformer` or `Normalizer`; Sparse → `MaxAbsScaler` or `Normalizer` |
| Special requirements | Need [0, 1] → `MinMaxScaler`; unit norm → `Normalizer`; preserve zero → `MaxAbsScaler`; custom range → `MinMaxScaler(feature_range=...)` |

## 4. Encoding categorical variables

**Summary.** Models need numbers, so categories must be converted. Two kinds of categorical variable:

* **Nominal** — no order (colour, brand, department).
* **Ordinal** — meaningful order (rating 1–5, Junior < Senior < Manager).

### Getting ready — toy categorical dataset

In [108]:
import numpy as np

# 2. Create sample categorical data with 20 records
np.random.seed(2024)  # for reproducibility
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame(
    {
        "Department": np.random.choice(
            categories, size=20),
        "Position": np.random.choice(
            ["Junior", "Senior", "Manager"], size=20),
        "Location": np.random.choice(
            ["NY", "SF", "LA", "CHI"], size=20),
    }
)

# 3. Display the DataFrame with categorical values
display(categorical_data)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA
6,C,Manager,SF
7,D,Manager,LA
8,B,Junior,LA
9,D,Manager,CHI


### `OneHotEncoder()`
One binary column per category; a 1 marks the record's category. Treats categories as independent (no fake order).

In [109]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_encoded_data = onehot_encoder.fit_transform(
    categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data,
    columns=onehot_encoder.get_feature_names_out()
)
onehot_encoded_df

,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
8,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [110]:
print("Original columns:", categorical_data.shape[1], "-> one-hot columns:", onehot_encoded_df.shape[1])
print("Each row has exactly one 1 per original feature:", (onehot_encoded_df.sum(axis=1) == categorical_data.shape[1]).all())
print("Learned categories_:", [list(c) for c in onehot_encoder.categories_])

Original columns: 3 -> one-hot columns: 11
Each row has exactly one 1 per original feature: True
Learned categories_: [['A', 'B', 'C', 'D'], ['Junior', 'Manager', 'Senior'], ['CHI', 'LA', 'NY', 'SF']]


### `LabelEncoder()`
Assigns an integer to every category (alphabetically). Compact, but the integers *look* ordered (A=0 < B=1 < C=2).

In [111]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoded_df = pd.DataFrame()
for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = (
        label_encoder.fit_transform(
            categorical_data[column]
        )
    )
label_encoded_df

,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1
6,2,1,3
7,3,1,1
8,1,0,1
9,3,1,0


> **Book correction / best practice.** `LabelEncoder` is designed for the **target** `y` (it accepts only 1-D input, which is why the book needs a `for` loop).
> For *input features* scikit-learn provides **`OrdinalEncoder`**, which handles many columns at once, lets you declare the true order
> via `categories=`, and works inside pipelines. The book's own note warns that integer codes can mislead models — for nominal data use one-hot.

**Extension — `OrdinalEncoder` with an explicit order, and handling unseen categories:**

In [112]:
from sklearn.preprocessing import OrdinalEncoder

ord_enc = OrdinalEncoder(categories=[["Junior", "Senior", "Manager"]])      # we state the true order
print(ord_enc.fit_transform(categorical_data[["Position"]])[:6].ravel(), "<- Junior=0, Senior=1, Manager=2")

# a category never seen in training would crash by default; handle_unknown makes it safe in production
ohe_safe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(categorical_data[["Location"]])
print(ohe_safe.transform(pd.DataFrame({"Location": ["BOS"]})))      # unseen city -> all zeros

# drop="first" removes one redundant column (avoids perfect multicollinearity in linear models)
print(OneHotEncoder(drop="first", sparse_output=False).fit(categorical_data[["Department"]]).get_feature_names_out())

[2. 2. 0. 2. 2. 1.] <- Junior=0, Senior=1, Manager=2
[[0. 0. 0. 0.]]
['Department_B' 'Department_C' 'Department_D']


### `ColumnTransformer()`
Applies different transformers to different columns — e.g. scale the numeric columns and one-hot encode the categorical ones — and
concatenates the result. First, the mixed dataset:

In [113]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# 2. Create sample mixed data with 20 records
np.random.seed(2024)  # for reproducibility
mixed_data = pd.DataFrame(
    {
        "Age": np.random.randint(25, 65, size=20),
        "Salary": np.round(np.random.normal(
            60000, 15000, size=20), 2),
        "Experience": np.random.randint(
            1, 20, size=20),
        "Department": np.random.choice(
            ["IT", "HR", "Sales", "Finance"], size=20),
        "Position": np.random.choice(
            ["Junior", "Senior", "Manager"], size=20),
    }
)

# 3. Display the DataFrame with mixed data
display(mixed_data)

,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior
6,60,49585.62,14,Finance,Junior
7,35,47992.58,17,HR,Manager
8,27,54833.02,9,Sales,Senior
9,57,93535.56,14,Sales,Senior


In [114]:
# 4. Initialize ColumnTransformer()
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),
         ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(),
         ["Department", "Position"]),
    ],
    remainder="passthrough",
)

# 5. Fit and transform the data
transformed_data = column_transformer.fit_transform(mixed_data)

# 6. Get feature names for the transformed columns
numeric_cols = [
    "Age_scaled", "Salary_scaled", "Experience_scaled"
]
categorical_cols = (
    column_transformer
    .named_transformers_["cat"]
    .get_feature_names_out(
        ["Department", "Position"]
    )
)

# 7. Create the transformed DataFrame
transformed_df = pd.DataFrame(
    transformed_data,
    columns=numeric_cols + list(categorical_cols)
)
transformed_df

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.045349,-0.305043,0.327303,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.727681,1.105091,1.262454,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636358,-1.581768,1.075424,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358300,-1.577167,-0.607848,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.023186,-0.565241,-0.420818,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.562482,0.852382,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
6,0.949309,-0.895798,0.701364,1.0,0.0,0.0,0.0,1.0,0.0,0.0
7,-0.897596,-0.991489,1.262454,0.0,1.0,0.0,0.0,0.0,1.0,0.0
8,-1.488606,-0.580596,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
9,0.727681,1.744195,0.701364,0.0,0.0,0.0,1.0,0.0,0.0,1.0


`remainder="passthrough"` keeps any column *not* listed (here there are none, since all five columns are listed; the default
`remainder="drop"` would discard them). `transformed_df` has 3 scaled + 4 department + 3 position = **10 columns**;
the book reuses it later as a ready-made dataset.

**Extension — the modern way to get column names automatically** (no manual `numeric_cols` list):

In [115]:
ct2 = ColumnTransformer(
    [("num", StandardScaler(), ["Age", "Salary", "Experience"]),
     ("cat", OneHotEncoder(sparse_output=False), ["Department", "Position"])],
    verbose_feature_names_out=False,
).set_output(transform="pandas")
ct2.fit_transform(mixed_data).head(3)

,Age,Salary,Experience,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.045349,-0.305043,0.327303,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.727681,1.105091,1.262454,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636358,-1.581768,1.075424,1.0,0.0,0.0,0.0,0.0,1.0,0.0


### Theory
* **One-hot encoding** represents a category with $k$ levels as $k$ binary indicators. It makes no assumption about order but raises dimensionality
  (high-cardinality features such as ZIP code create huge sparse matrices — hence `sparse_output=True`, the default, and options such as `min_frequency`).
  With an intercept in a linear model, the $k$ columns sum to 1 (the *dummy-variable trap*); `drop="first"` avoids that.
* **Ordinal / label encoding** gives one integer per category. Fine for true ordinal data and for tree models (which can split on thresholds);
  harmful for nominal data in distance/linear models because the codes imply distances (A is "closer" to B than to D).
* **ColumnTransformer** is the standard way to preprocess heterogeneous tables. Always put it **inside** a `Pipeline` so it is fit on training data only.
* Choosing: nominal → one-hot; ordinal → `OrdinalEncoder(categories=...)`; target labels → `LabelEncoder`.

### Hands-on workflow (from the book)
Identify categorical features → decide nominal vs ordinal → apply the matching encoder → merge with numeric features (`ColumnTransformer`) → evaluate how each encoding changes performance.

## 5. Introduction to pipelines in scikit-learn

**Summary.** A `Pipeline` is a list of `(name, transformer)` steps ending optionally in an estimator. Benefits listed in the book:
*sequential execution, code simplification, consistency (less leakage), easier hyper-parameter tuning, modularity*.

General syntax:
```python
pipeline = Pipeline([("name of step", transformer),
                     ("name of step", transformer), ...,
                     ("name of step", estimator)])
```

### How to do it…
Handle missing values, then scale. The book uses `transformed_df` from the previous section: all columns but the last are the
features `X`, the last column (`Position_Senior`) is the "target" `y`.

In [116]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 2. First, separate the features and target (assuming the last column is the target)
X = transformed_df.iloc[:, :-1]  # all columns except last
y = transformed_df.iloc[:, -1]   # last column

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

# 4. Create a pipeline
pipeline = Pipeline([
    (
        "imputer", SimpleImputer(strategy="mean")
    ),  # handle missing values
    (
        "scaler", StandardScaler()
    ),  # scale the features
])

# 5. Fit and transform the training data
X_train_transformed = pipeline.fit_transform(X_train)

# 6. Transform the test data
X_test_transformed = pipeline.transform(X_test)

# 7. Create DataFrames with the transformed data (to preserve column names)
X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=X_train.columns,
    index=X_train.index
)
X_test_transformed = pd.DataFrame(
    X_test_transformed,
    columns=X_test.columns,
    index=X_test.index
)
print("train:", X_train_transformed.shape, " test:", X_test_transformed.shape)
print("target used:", y.name)

train: (16, 9)  test: (4, 9)
target used: Position_Senior


> **Comment on the book's example.** The "target" here is a one-hot column (`Position_Senior`) of already-encoded, randomly generated data,
> so the example only demonstrates *mechanics* — there is no real signal to predict. Also note that the data has no NaNs left, so the imputer does nothing;
> it is there to show where imputation belongs in a pipeline.

**Inspect what the pipeline learned** — the test set is scaled with the *training* statistics, so its mean is *not* exactly 0:

In [117]:
print("train means:", X_train_transformed.mean().round(3).values[:4], "...")
print("test  means:", X_test_transformed.mean().round(3).values[:4], "...  <- not 0, as intended (no refit on test)")
print("pipeline step names:", list(pipeline.named_steps))
print("learned scaler mean_:", pipeline.named_steps["scaler"].mean_[:3].round(3))

train means: [ 0. -0.  0. -0.] ...
test  means: [0.961 0.078 0.503 0.577] ...  <- not 0, as intended (no refit on test)
pipeline step names: ['imputer', 'scaler']
learned scaler mean_: [-0.196 -0.017 -0.094]


### Splitting before data transformations
The book's argument (summarised): split **first**, fit on **train** only, then `transform` the test set with those *same* parameters.
This (1) prevents leakage, (2) gives a realistic estimate on unseen data, (3) keeps train/test consistent, (4) simplifies the workflow and (5) reduces overfitting to dataset quirks.

**Extension — leakage in numbers.** Feature selection done on *all* the data before cross-validation vs inside a pipeline, on a dataset with many
features and few samples (where leakage hurts most). The same logic applies to scalers and imputers, just with a smaller effect:

In [118]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

Xs, ys = make_classification(n_samples=150, n_features=500, n_informative=5, random_state=0)  # many features, few samples
from sklearn.feature_selection import SelectKBest, f_classif

# WRONG: feature selection sees ALL labels before cross-validation splits -> leakage
Xsel = SelectKBest(f_classif, k=10).fit_transform(Xs, ys)
wrong = cross_val_score(LogisticRegression(max_iter=1000), Xsel, ys, cv=5).mean()

# RIGHT: selection happens inside each training fold
right = cross_val_score(make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression(max_iter=1000)), Xs, ys, cv=5).mean()
print(f"leaky estimate   : {wrong:.3f}")
print(f"pipeline estimate: {right:.3f}   <- honest")

leaky estimate   : 0.787
pipeline estimate: 0.667   <- honest


The leaky number is higher — the model "saw" label information through the selection step. This is the bug pipelines prevent.

### Visualizing pipelines
`set_config(display="diagram")` makes scikit-learn estimators render as an interactive diagram in notebooks (this is the default in recent versions).

In [119]:
from sklearn import set_config

set_config(display="diagram")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['Age_scaled','Salary_scaled','Experience_scaled',...,'Department_Sales', 'Position_Junior','Position_Manager']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" f

**Extension — a complete pipeline: `ColumnTransformer` + model, tuned with `GridSearchCV`.**
Step names let you tune any step with `stepname__param`.

In [120]:
from sklearn.model_selection import GridSearchCV

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
     ["Age", "Experience"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["Department", "Position"]),
])
full = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

# demonstration target: "is Salary above the median?" (random data -> expect ~chance, the point is the mechanics)
y_demo = (mixed_data["Salary"] > mixed_data["Salary"].median()).astype(int)
X_demo = mixed_data.drop(columns="Salary")                    # the target's source column must NOT be a feature
search = GridSearchCV(full, {"model__C": [0.1, 1, 10], "prep__num__impute__strategy": ["mean", "median"]}, cv=3)
search.fit(X_demo, y_demo)
print("best params:", search.best_params_)
full

best params: {'model__C': 10, 'prep__num__impute__strategy': 'mean'}


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output 

### Theory
* **Pipeline mechanics.** `fit(X, y)`: for every step except the last call `fit_transform`, feed the output forward; the final step gets `fit`.
  `predict(X)`: `transform` through all steps, then `predict`. Because the CV splitter wraps the *whole* pipeline,
  every fold re-learns imputer/scaler statistics from its own training part.
* **Leakage types:** *target leakage* (a feature that encodes the answer) and *train-test contamination* (preprocessing fit on all data).
  Splitting first plus `Pipeline` removes the second kind.
* `make_pipeline(...)` auto-names steps; `Pipeline.named_steps` / `pipeline[0]` access steps; `set_output(transform="pandas")` keeps column names.
* **Caveat:** a `Pipeline` only protects you if the *split* (or the CV object) wraps it. Fitting any transformer on the full dataset first defeats the purpose.

### Hands-on workflow (from the book)
Decide required preprocessing → choose an algorithm → chain with `Pipeline` → fit on training data, predict on new data → assess with suitable metrics.

## 6. Feature engineering

**Summary.** Feature engineering = **feature extraction** (create new features from existing ones) + **feature selection**
(keep only the informative ones). Good features let simpler models generalise better. Methods are grouped as
*filter* (statistical properties, model-free), *wrapper* (evaluate subsets by training models) and *hybrid* (a mix).
The book reuses `X_train_transformed` from the pipeline section.

### Common mathematical transformations — `PolynomialFeatures()`
Generates polynomial and interaction terms. With $n$ features and `degree=2` it creates $1,\,x_i,\,x_i^2,\,x_ix_j$.

In [121]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
    poly_features,
    columns = poly.get_feature_names_out(
        X_train_transformed.columns
    )
)
poly_features_df

,1,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,...,Department_IT^2,Department_IT Department_Sales,Department_IT Position_Junior,Department_IT Position_Manager,Department_Sales^2,Department_Sales Position_Junior,Department_Sales Position_Manager,Position_Junior^2,Position_Junior Position_Manager,Position_Manager^2
0,1.0,0.834298,-0.867451,-1.559779,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,-1.0,...,4.333333,-1.612452,-1.201850,-2.081666,0.600000,0.447214,0.774597,0.333333,0.577350,1.0
1,1.0,-0.689202,1.282818,-0.150946,-0.577350,2.081666,-0.480384,-0.774597,-0.577350,1.0,...,0.230769,0.372104,0.277350,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
2,1.0,0.544107,-1.173357,-1.761041,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,1.0,...,4.333333,-1.612452,-1.201850,2.081666,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
3,1.0,-1.342131,0.821525,-0.150946,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,0.230769,-0.620174,0.277350,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.577350,1.0
4,1.0,1.414679,0.611115,0.654101,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,1.0,...,0.230769,-0.620174,0.277350,-0.480384,1.666667,-0.745356,1.290994,0.333333,-0.577350,1.0
5,1.0,-1.414679,-1.479801,1.257887,1.732051,-0.480384,-0.480384,-0.774597,-0.577350,1.0,...,0.230769,0.372104,0.277350,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
6,1.0,1.269583,0.576819,0.452839,1.732051,-0.480384,-0.480384,-0.774597,1.732051,-1.0,...,0.230769,0.372104,-0.832050,0.480384,0.600000,-1.341641,0.774597,3.000000,-1.732051,1.0
7,1.0,0.544107,-1.475450,-0.553470,-0.577350,-0.480384,2.081666,-0.774597,1.732051,-1.0,...,4.333333,-1.612452,3.605551,-2.081666,0.600000,-1.341641,0.774597,3.000000,-1.732051,1.0
8,1.0,0.906845,1.060444,1.459148,-0.577350,-0.480384,-0.480384,1.290994,1.732051,-1.0,...,0.230769,-0.620174,-0.832050,0.480384,1.666667,2.236068,-1.290994,3.000000,-1.732051,1.0
9,1.0,0.906845,1.664674,0.855363,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,0.230769,-0.620174,0.277350,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.577350,1.0


In [122]:
print("input features:", X_train_transformed.shape[1], "-> polynomial (deg 2) features:", poly_features_df.shape[1])
import math
print("formula  C(n+d, d) = C(9+2, 2) =", math.comb(9 + 2, 2))
print(list(poly_features_df.columns[:8]))

input features: 9 -> polynomial (deg 2) features: 55
formula  C(n+d, d) = C(9+2, 2) = 55
['1', 'Age_scaled', 'Salary_scaled', 'Experience_scaled', 'Department_Finance', 'Department_HR', 'Department_IT', 'Department_Sales']


### `KBinsDiscretizer()`
Turns a continuous variable into discrete bins ("buckets"). `n_bins` sets the number of buckets and `strategy` the boundaries:
`uniform` (equal width), `quantile` (equal count per bin) or `kmeans` (bins around 1-D k-means centres).

In [123]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(
    n_bins=3, encode="ordinal", strategy="uniform")
binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(
    binned_data, columns=X_train_transformed.columns)
binned_df

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
2,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0
3,0.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,2.0,0.0,2.0
5,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,2.0
6,2.0,1.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0
7,2.0,0.0,1.0,0.0,0.0,2.0,0.0,2.0,0.0
8,2.0,2.0,2.0,0.0,0.0,0.0,2.0,2.0,0.0
9,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0


Compare the three strategies on one continuous feature (bin edges and how many samples fall in each bin):

In [124]:
col = X_train_transformed[["Salary_scaled"]]
for strat in ("uniform", "quantile", "kmeans"):
    kb = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy=strat).fit(col)
    counts = np.bincount(kb.transform(col).ravel().astype(int), minlength=3)
    print(f"{strat:9s} edges={kb.bin_edges_[0].round(2)}  samples per bin={counts}")

uniform   edges=[-1.48 -0.43  0.62  1.66]  samples per bin=[7 4 5]
quantile  edges=[-1.48 -0.53  0.61  1.66]  samples per bin=[5 5 6]
kmeans    edges=[-1.48 -0.36  0.72  1.66]  samples per bin=[7 4 5]


### Techniques for selecting relevant features — `RFE()`
*Recursive Feature Elimination*: train an estimator on all features, drop the least important, retrain, and repeat until the desired number remain.
With `n_features_to_select=1` the algorithm keeps going until one feature is left, so `ranking_` becomes a full ranking (1 = last survivor = most important).
`RFECV()` adds cross-validation to pick the number of features automatically.

In [125]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(
    estimator=LinearRegression(),
    n_features_to_select=1
)
rfe.fit(X_train_transformed, y_train)
rfe.ranking_

array([7, 5, 4, 3, 6, 9, 8, 2, 1])

The book's printed result is `array([5, 4, 8, 6, 9, 3, 7, 2, 1])`. Map the ranks to names:

In [126]:
print("book's ranking   :", [5, 4, 8, 6, 9, 3, 7, 2, 1])
print("our ranking      :", rfe.ranking_.tolist())
pd.Series(rfe.ranking_, index=X_train_transformed.columns).sort_values().to_frame("rank (1 = best)")

book's ranking   : [5, 4, 8, 6, 9, 3, 7, 2, 1]
our ranking      : [7, 5, 4, 3, 6, 9, 8, 2, 1]


,rank (1 = best)
Position_Manager,1
Position_Junior,2
Department_Finance,3
Experience_scaled,4
Salary_scaled,5
Department_HR,6
Age_scaled,7
Department_Sales,8
Department_IT,9


**Why our ranking differs from the book's (and why that is fine).** Only the *top two* positions agree (features 8 and 7 = `Position_Manager`, `Position_Junior`).
The reason is the toy example itself: the target is `Position_Senior`, and one-hot columns always satisfy
`Position_Junior + Position_Manager + Position_Senior = 1`, so **the target is an exact linear function of two features**
(`y = 1 − Junior − Manager`). A linear model therefore gives those two a large coefficient and every other feature a coefficient that is
numerically zero (≈ 1e-15, see the next output). Ranking numbers that are all "zero plus rounding noise" depends on the platform, library version and BLAS,
so the ordering of the remaining seven features is arbitrary. This is a good illustration of **target leakage** (Section 1) and of why
you should interpret rankings only when the signal is real.

### `SelectFromModel()`
Keeps features whose importance (here `|coef_|` of a `LinearRegression`) is above a `threshold` (`'mean'` = mean importance).
Works with any estimator exposing `coef_` or `feature_importances_`.

In [127]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

selector = SelectFromModel(
    estimator=LinearRegression(),
    prefit=False,
    # Use mean of feature importances as threshold:
    threshold='mean'
)
selector.fit(X_train_transformed, y_train)

# 4. Get the selected features
selected_features_mask = selector.get_support()

# 5. Get the feature names that were selected
selected_features = X_train_transformed.columns[
    selected_features_mask
].tolist()
print("Selected:", selected_features)

# 6. Print feature importance scores and selection status
feature_importance = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Importance': selector.estimator_.coef_,
    'Selected': selected_features_mask
})
feature_importance.sort_values(
    'Importance', key=abs, ascending=False)

Selected: ['Position_Junior', 'Position_Manager']


,Feature,Importance,Selected
8,Position_Manager,-5.000000e-01,True
7,Position_Junior,-4.330127e-01,True
2,Experience_scaled,-1.103284e-15,False
1,Salary_scaled,1.054712e-15,False
4,Department_HR,-4.649059e-16,False
3,Department_Finance,4.614364e-16,False
6,Department_Sales,-2.810252e-16,False
0,Age_scaled,2.078464e-16,False
5,Department_IT,-2.046974e-16,False


**Extension — a filter method for comparison, and picking the number of features with `RFECV`.**
`SelectKBest` scores each feature independently (no model training = *filter*), RFE/`RFECV` trains models repeatedly (*wrapper*).
Here I use the *Diabetes* data, which has real signal (the book's toy target has none).

In [128]:
from sklearn.datasets import load_diabetes
from sklearn.feature_selection import SelectKBest, f_regression, RFECV

Xd, yd = load_diabetes(return_X_y=True, as_frame=True)

filt = SelectKBest(f_regression, k=4).fit(Xd, yd)                      # filter
print("Filter  (SelectKBest, F-test) keeps:", list(Xd.columns[filt.get_support()]))

rfecv = RFECV(LinearRegression(), cv=5, scoring="r2").fit(Xd, yd)       # wrapper + CV
print("Wrapper (RFECV) keeps             :", list(Xd.columns[rfecv.support_]), f"(optimal n = {rfecv.n_features_})")

Filter  (SelectKBest, F-test) keeps: ['bmi', 'bp', 's4', 's5']
Wrapper (RFECV) keeps             : ['sex', 'bmi', 'bp', 's1', 's2', 's5'] (optimal n = 6)


### Theory
* **Feature extraction.** `PolynomialFeatures` lets a *linear* model fit curves and interactions: a linear model on $[1,x,x^2]$ is quadratic in $x$.
  The number of terms grows as $\binom{n+d}{d}$, so degree and feature count explode quickly (55 terms from 9 features at degree 2) — risk of overfitting; usually combined with regularisation or selection.
* **Discretisation** (`KBinsDiscretizer`) makes a model piecewise-constant; can capture non-linear effects and reduce outlier influence at the price of losing within-bin detail.
  `uniform` is outlier-sensitive; `quantile` balances counts; `kmeans` follows clusters.
* **Filter methods** (variance threshold, correlation, $\chi^2$, ANOVA F, mutual information): fast, model-agnostic, ignore feature interactions.
  **Wrapper methods** (RFE): use a model's performance/importances to search subsets — more accurate, more expensive, can overfit the selection estimator.
  **Embedded methods** (`SelectFromModel`, Lasso): selection happens during training through the importances/coefficients.
* **RFE** removes the weakest feature each round (`step` features at a time), so correlated features are handled better than by a one-shot ranking.
* **Coefficient-based importance is only comparable on standardised features** — the book applies it to scaled data for that reason.
  With a plain `LinearRegression` the importances are noisy when features are correlated (here: one-hot columns that sum to 1).
* **Selection must live inside the CV loop / pipeline**, as the leakage experiment in Section 5 showed.
* Remember the trade-off in the book: as model complexity increases, interpretability decreases.

### Hands-on workflow (from the book)
Understand the dataset → derive new variables (maths/domain knowledge) → apply `RFE`/`SelectFromModel` → train models on original vs engineered data and compare.

## 7. Practical exercise — a comprehensive pipeline on California Housing

**Task from the book:** (1) load California Housing, (2) split, (3) build a pipeline with **at least three steps** whose last step is an estimator,
(4) fit, (5) evaluate on the test set. The book leaves a stub and ships a separate "(Solution)" notebook — this is my own solution.

The dataset has 20 640 districts and 8 numeric features; the target `MedHouseVal` is the **median** house value in units of \$100 000.

> **Book correction.** The book says "9 features" (it counts the target) and "average home price per 100,000 homes";
> the target is the median house value of a block group, expressed in \$100 000s.

### 1–2. Load and split

In [129]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
X_h, y_h = housing.data, housing.target
print(X_h.shape, "| target:", y_h.name)
display(X_h.describe().T[["mean", "std", "min", "max"]].round(2))

X_tr, X_te, y_tr, y_te = train_test_split(X_h, y_h, test_size=0.2, random_state=2024)
print("train:", X_tr.shape, "test:", X_te.shape)

(20640, 8) | target: MedHouseVal


,mean,std,min,max
MedInc,3.87,1.90,0.50,15.00
HouseAge,28.64,12.59,1.00,52.00
AveRooms,5.43,2.47,0.85,141.91
AveBedrms,1.10,0.47,0.33,34.07
Population,1425.48,1132.46,3.00,35682.00
AveOccup,3.07,10.39,0.69,1243.33
Latitude,35.63,2.14,32.54,41.95
Longitude,-119.57,2.00,-124.35,-114.31


train: (16512, 8) test: (4128, 8)


The features sit on very different scales (`Population` up to 35 682 vs `AveBedrms` around 1), and `AveRooms`, `AveOccup`, `Population` have extreme
outliers — a good case for scaling and for the book's decision guide.

### 3–5. Pipeline, fit, evaluate
I build the pipeline **one step at a time** and record the test score after each addition, so you can see what every step buys (an *ablation*).
The estimator is always `Ridge`; only the preprocessing changes.

In [130]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, QuantileTransformer, PolynomialFeatures
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def scores(model):
    pred = model.predict(X_te)
    return {"R2": r2_score(y_te, pred),
            "RMSE": np.sqrt(mean_squared_error(y_te, pred)),
            "MAE": mean_absolute_error(y_te, pred)}

impute = ("imputer", SimpleImputer(strategy="median"))                          # 1. handle missing values
variants = {
    "0. LinearRegression only (baseline)": LinearRegression(),
    "1. impute + scale + Ridge":            Pipeline([impute, ("scaler", StandardScaler()), ("model", Ridge(10))]),
    "2. + PolynomialFeatures(2)":           Pipeline([impute, ("scaler", StandardScaler()),
                                                      ("poly", PolynomialFeatures(2, include_bias=False)), ("model", Ridge(10))]),
    "3. quantile scaler instead of Standard": Pipeline([impute, ("scaler", QuantileTransformer(output_distribution="normal")),
                                                      ("poly", PolynomialFeatures(2, include_bias=False)), ("model", Ridge(10))]),
    "4. + Lasso feature selection":         Pipeline([impute, ("scaler", QuantileTransformer(output_distribution="normal")),
                                                      ("poly", PolynomialFeatures(2, include_bias=False)),
                                                      ("selector", SelectFromModel(Lasso(alpha=0.001, max_iter=20000), threshold="median")),
                                                      ("model", Ridge(10))]),
}
rows = {}
for name, est in variants.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        est.fit(X_tr, y_tr)
    rows[name] = scores(est)
ablation = pd.DataFrame(rows).T.round(3)
ablation

,R2,RMSE,MAE
0. LinearRegression only (baseline),0.599,0.741,0.535
1. impute + scale + Ridge,0.599,0.741,0.535
2. + PolynomialFeatures(2),0.647,0.694,0.467
3. quantile scaler instead of Standard,0.690,0.651,0.462
4. + Lasso feature selection,0.684,0.657,0.470


Reading the table:
* **Scaling alone does not change plain linear regression's accuracy** (the baseline and step 1 are close) — it matters for Ridge's penalty and for the next steps.
* **`PolynomialFeatures`** gives the biggest jump: housing prices depend non-linearly on income and *location* (latitude × longitude interactions).
* **Quantile-to-normal scaling** helps further because `AveRooms`, `AveOccup`, `Population` are very skewed with extreme outliers, exactly the case the book's decision guide
  (Figure 2.17) assigns to `QuantileTransformer`/`PowerTransformer`.
* **Feature selection** removes about half of the 44 polynomial terms for a small change in accuracy — simpler model, similar performance. A simpler model with the same
  score is the book's stated goal for selection. (Selection does not always improve the score; it is a trade-off with simplicity.)

Final pipeline (step 4) and a look at its structure:

In [131]:
comprehensive = variants["4. + Lasso feature selection"]
print("features after poly :", comprehensive.named_steps["poly"].n_output_features_)
print("features kept       :", comprehensive.named_steps["selector"].get_support().sum())
comprehensive

features after poly : 44
features kept       : 22


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['MedInc','HouseAge','AveRooms',...,'AveOccup','Latitude','Longitude']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",

Now tune a few hyper-parameters together with cross-validation (`stepname__param`), without ever touching the test set:

In [132]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    comprehensive,
    {"poly__degree": [2, 3],
     "selector__threshold": ["median", "mean"],
     "model__alpha": [1.0, 10.0, 100.0]},
    cv=3, scoring="r2", n_jobs=-1,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    grid.fit(X_tr, y_tr)
print("best params :", grid.best_params_)
print(f"CV R2       : {grid.best_score_:.3f}")
print("TEST scores :", {k: round(v, 3) for k, v in scores(grid.best_estimator_).items()})

best params : {'model__alpha': 10.0, 'poly__degree': 3, 'selector__threshold': 'median'}
CV R2       : 0.728
TEST scores : {'R2': 0.723, 'RMSE': np.float64(0.616), 'MAE': 0.432}


### Theory — evaluating a regression model (the part the book asks you to research)
For predictions $\hat y_i$ and truths $y_i$ over $n$ test samples:
* **MSE** $=\dfrac1n\sum(y_i-\hat y_i)^2$ and **RMSE** $=\sqrt{\text{MSE}}$ — same unit as the target (\$100 000s here); penalises large errors heavily.
* **MAE** $=\dfrac1n\sum|y_i-\hat y_i|$ — average absolute error; more robust to outliers.
* **R²** $=1-\dfrac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$ — fraction of variance explained (1 = perfect, 0 = no better than predicting the mean).
* Always evaluate on the **held-out test set** (or cross-validation), never on the data the model was fit on.
* **Why this pipeline works:** scaling makes the Lasso/Ridge penalties fair across features; the quantile transform tames skewed, outlier-ridden features;
  polynomial terms let a linear model capture non-linear effects (e.g. location × income); the Lasso-based selector prunes the 44 polynomial terms;
  Ridge shrinks the remaining coefficients to control variance. Ridge and Lasso are studied in Chapter 5.
* **A linear pipeline has a ceiling.** Tree ensembles (Chapter 8) usually reach higher R² on this dataset with *less* preprocessing. The point of this exercise is the
  *workflow*, not the highest score.

## Chapter takeaways
1. **Data quality first.** Missing values, outliers, scale differences and categorical variables each have a dedicated scikit-learn tool.
2. **Imputation:** `SimpleImputer` (fast baseline) < `KNNImputer` < `IterativeImputer` (uses relationships between features, costliest).
3. **Scaling** matters for distance/gradient/regularised/PCA models and not for trees. Pick the scaler from outliers and distribution (Figure 2.17).
4. **Encoding:** one-hot for nominal, ordinal encoding with a declared order for ordinal data; use `ColumnTransformer` for mixed tables.
5. **Always split first and use a `Pipeline`** — it prevents leakage and makes tuning and deployment a one-object affair.
6. **Feature engineering** creates (`PolynomialFeatures`, `KBinsDiscretizer`) and prunes (`RFE`, `SelectFromModel`) features; selection belongs inside the CV loop.

### Book corrections noted in this notebook
| Where | Book says | Actually |
|---|---|---|
| `KNNImputer` | fills with the "majority label" of neighbours | fills with the **mean** of neighbours' values |
| Imputation guide | `>5 %` missing → `SimpleImputer` | clearly means **<5 %** |
| `LabelEncoder` on features | used for input columns | designed for the target; use `OrdinalEncoder` for features |
| Closing paragraph | "feature engineering in Chapter 1 … dimensionality reduction in Chapter 2" | feature engineering is in **Chapter 2**, dimensionality reduction in **Chapter 3** |
| California Housing | "9 features", "average home price per 100,000 homes" | 8 features (+ target); **median** house value in \$100 000s |

*Next: Chapter 3 — Dimensionality Reduction Techniques.*